# S3 STARE-PODS Demo — AWS S3 + RDS Postgres

Cloud counterpart of `local_starepods_examples.ipynb`. Runs the same flow against real **AWS S3** (Parquet partitions) and a real **RDS Postgres** `PodsMetadata` table.

**Core workflow**
1. **Ingest granules from four instruments** (GMI, SSMIS, AMSR2, ATMS) → S3 Parquet + RDS metadata (four instruments so the overlap analytics in step 10 can show 2-, 3- and 4-way rendezvous)
2. **Find intersecting data** via STARE SIDs + RDS (bbox filter optional; default loads the full granule)
3. **Download intersecting Parquet partitions** from S3
4. **Reconstitute HDF5** (S1 + S2 scans)
5. **Structure comparison** — reconstituted vs original
6. **RDS metadata verification**

**Temporal features**

7. **Temporal catalog** — every chunk carries `[t_start, t_end]` + podcode
8. **Period-filtered load** — data-level `[t_start, t_end]` overlap
9. **VCF temporal roll-up** — union range per pod, on the fly
10. **Multi-instrument overlap analytics** — 2-, 3- and 4-way rendezvous

> The S3/RDS temporal loaders read the **shared** `PodsMetadata` catalog — every ingest in the RDS table, not only this demo's granule (filtered by instrument). That is the production query surface, so the temporal counts reflect the whole catalog, unlike the local demo's fresh isolated SQLite.

**Requires** `starepandas/.config` with AWS + RDS credentials. Sample granules default to the in-repo GMI + SSMIS pair plus the four rendezvous granules; override the pair with `STAREPODS_SAMPLE_GRANULE` / `STAREPODS_SAMPLE_GRANULE_SSMIS`.

In [1]:
#import subprocess, sys
#subprocess.check_call([sys.executable, "-m", "pip", "install", "-e",
#                       "/Users/thatdaihaiton/Workspace/STARE/STAREPandas/stare_demo_and_construct_parallel",
#                       "-q"])

In [2]:
import os
import time
import h5py
from starepandas.demo_lib import StarePodsDemo
from starepandas.staredataframe import _ensure_rds_db_and_table

## Configuration

Edit these paths and parameters before running.

In [3]:
import pandas as pd
import starepandas

# AWS + RDS credentials. Resolved relative to the installed package so it works
# from any cwd (the .config lives next to the starepandas package).
CONFIG_PATH = os.path.join(os.path.dirname(os.path.abspath(starepandas.__file__)), ".config")

# Resolve the sample granules from the in-repo test-data dir so the notebook is
# safe to run anywhere (no dependency on an external sample directory). Override
# the pair with the STAREPODS_SAMPLE_GRANULE* env vars.
_REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(starepandas.__file__)))
_GRANULE_DIR = os.path.join(_REPO_ROOT, "tests", "data", "granules")

GRANULE_FILE = os.environ.get(
    "STAREPODS_SAMPLE_GRANULE",
    os.path.join(_GRANULE_DIR,
                 "1C.GPM.GMI.XCAL2016-C.20250101-S112952-E130304.061572.V07B.HDF5"),
)

# GMI + SSMIS are a co-located pair (both 2025-01-01, concurrent orbits) whose
# ground tracks cross within ~3 min in 42 shared pods — the tightest rendezvous
# in the demo data.
SSMIS_GRANULE_FILE = os.environ.get(
    "STAREPODS_SAMPLE_GRANULE_SSMIS",
    os.path.join(_GRANULE_DIR,
                 "1C.F18.SSMIS.XCAL2021-V.20250101-S112813-E131004.078441.V07B.HDF5"),
)

# A pair only ever fills the n=2 column of the slide-9 table. These four
# granules — one per instrument, all later the same day — are a verified
# **4-way** rendezvous: over pods q03200 and q03203 the passes arrive
# SSMIS 21:29 -> AMSR2 21:46 -> ATMS 21:47 -> GMI 22:13, i.e. all four within
# ~45 min. Ingesting them alongside the pair populates every cell of the
# slide-8 matrix and the n=2/3/4 columns of the slide-9 table from real data.
RENDEZVOUS_GRANULES = [
    ("SSMIS", os.path.join(_GRANULE_DIR,
              "1C.F18.SSMIS.XCAL2021-V.20250101-S195732-E213923.078446.V07B.HDF5")),
    ("AMSR2", os.path.join(_GRANULE_DIR,
              "1C.GCOMW1.AMSR2.XCAL2016-V.20250101-S201914-E215806.067167.V07A.HDF5")),
    ("ATMS",  os.path.join(_GRANULE_DIR,
              "1C.NOAA21.ATMS.XCAL2023-V.20250101-S201707-E215835.011117.V07A.HDF5")),
    ("GMI",   os.path.join(_GRANULE_DIR,
              "1C.GPM.GMI.XCAL2016-C.20250101-S204910-E222221.061578.V07B.HDF5")),
]

INSTRUMENTS = ["GMI", "SSMIS", "AMSR2", "ATMS"]

# Coincidence window for step 10. The four passes above span ~45 min, so a
# narrower window still shows 2- and 3-way rendezvous but no 4-way.
OVERLAP_DT = pd.Timedelta(minutes=45)

# The RDS catalog is shared with every other ingest, so step 10 scopes its read
# to the two windows this demo actually wrote — the tight pair and the 4-way —
# rather than sweeping the whole table. Both push into SQL.
DEMO_WINDOWS = [
    (pd.Timestamp("2025-01-01 11:00"), pd.Timestamp("2025-01-01 13:30")),
    (pd.Timestamp("2025-01-01 20:00"), pd.Timestamp("2025-01-01 23:00")),
]

# Step 8 splits the two GMI passes at this instant, which lies in the ~7 h gap
# between them (they cover 11:29-13:03 and 20:49-22:22).
PASS_SPLIT = pd.Timestamp("2025-01-01T16:00:00")

# S3 root where Parquet partitions and RDS metadata for this demo live.
S3_PREFIX = "s3://zarrpods/gmi-demo-parquet"

# STARE partition level used for both ingestion and bbox → SIDs lookup.
# Capped at MAX_PARTITION_LEVEL = 4 (~256 cells/granule), the regime
# where each Parquet partition is multi-MB — ideal for S3.
STARE_LEVEL = 4

# Bounding box filter — set to None to reconstitute the full granule
# (matching local_starepods_examples.ipynb), or e.g. (115, -30, 120, -25)
# to restrict to SW Australia / Perth.
BBOX = None   # full granule, no spatial filter — mirrors the local demo

DATASETS = ["GMI_S1", "GMI_S2"]

OUTPUT_HDF5 = "/tmp/gmi_s3_reconstituted.h5"

# Set to True to wipe S3_PREFIX (S3 objects + RDS metadata rows) before
# ingesting. Mirrors the local demo's CLEAN_BEFORE_RUN flag — prevents
# duplicate RDS rows on re-runs. Keep True unless you intentionally
# want to append more granules under the same prefix.
CLEAN_BEFORE_RUN = True

print(f"Granule  : {os.path.basename(GRANULE_FILE)}")
print(f"SSMIS    : {os.path.basename(SSMIS_GRANULE_FILE)}")
for _instrument, _path in RENDEZVOUS_GRANULES:
    print(f"{_instrument:9s}: {os.path.basename(_path)}")
print(f"Datasets : {DATASETS}")
print(f"BBox     : {BBOX}  (None = full granule)")
print(f"S3 root  : {S3_PREFIX}")
print(f"Clean    : {CLEAN_BEFORE_RUN}")

Granule  : 1C.GPM.GMI.XCAL2016-C.20250101-S112952-E130304.061572.V07B.HDF5
SSMIS    : 1C.F18.SSMIS.XCAL2021-V.20250101-S112813-E131004.078441.V07B.HDF5
SSMIS    : 1C.F18.SSMIS.XCAL2021-V.20250101-S195732-E213923.078446.V07B.HDF5
AMSR2    : 1C.GCOMW1.AMSR2.XCAL2016-V.20250101-S201914-E215806.067167.V07A.HDF5
ATMS     : 1C.NOAA21.ATMS.XCAL2023-V.20250101-S201707-E215835.011117.V07A.HDF5
GMI      : 1C.GPM.GMI.XCAL2016-C.20250101-S204910-E222221.061578.V07B.HDF5
Datasets : ['GMI_S1', 'GMI_S2']
BBox     : None  (None = full granule)
S3 root  : s3://zarrpods/gmi-demo-parquet
Clean    : True


## Step 1 — Ingest GMI, SSMIS, AMSR2 and ATMS granules → S3 Parquet + RDS

In [4]:
%%time
demo = StarePodsDemo(aws_config_path=CONFIG_PATH)

s3_paths = demo.ingest_granules(
    data_path=GRANULE_FILE,
    instrument="GMI",
    s3_prefix=S3_PREFIX,
    level=STARE_LEVEL,
    clean_before_run=CLEAN_BEFORE_RUN,
)
# Every later granule appends — clean_before_run=False so it does NOT wipe
# the data already written to the same prefix.
ssmis_paths = demo.ingest_granules(
    data_path=SSMIS_GRANULE_FILE,
    instrument="SSMIS",
    s3_prefix=S3_PREFIX,
    level=STARE_LEVEL,
    clean_before_run=False,
)
print(f"GMI  : stored {len(s3_paths)} dataset path(s)")
print(f"SSMIS: stored {len(ssmis_paths)} dataset path(s)")

for instrument, path in RENDEZVOUS_GRANULES:
    paths = demo.ingest_granules(
        data_path=path,
        instrument=instrument,
        s3_prefix=S3_PREFIX,
        level=STARE_LEVEL,
        clean_before_run=False,
    )
    print(f"{instrument:5s}: stored {len(paths)} dataset path(s)  ({os.path.basename(path)})")

# Granule basename — used as a substring filter on group_path. Note: as of
# the quaternary pod-code layout (2026-06-14) the S3 layout is FLAT and the
# granule basename is embedded in the chunk *filename*, bracketed by '-':
#   <S3_PREFIX>/<podcode>-<granule_basename>-<dataset>.parquet
# So the old startswith(S3_PREFIX + '/' + basename) scoping no longer matches.
# We use a substring match on the basename instead — which also keeps steps
# 2-5 scoped to this granule now that two GMI granules are ingested.
granule_basename = os.path.splitext(os.path.basename(GRANULE_FILE))[0]
granule_path_marker = f"-{granule_basename}-"   # matches the filename-embedded span

INFO:starepandas.ingest:clean_before_run=True → wiping s3://zarrpods/gmi-demo-parquet on S3 + RDS first


INFO:starepandas.ingest:clean_s3_prefix(s3://zarrpods/gmi-demo-parquet): deleted 1982 RDS row(s), 1982 S3 object(s)


INFO:starepandas.ingest:Ingesting GMI granules from /Users/thatdaihaiton/Workspace/STARE/STAREPandas/stare_pods_aws_parallel/tests/data/granules/1C.GPM.GMI.XCAL2016-C.20250101-S112952-E130304.061572.V07B.HDF5


INFO:starepandas.ingest:Found 1 GMI file(s)


INFO:starepandas.ingest:Processing 1C.GPM.GMI.XCAL2016-C.20250101-S112952-E130304.061572.V07B.HDF5


...

  Progress: 150/243 partitions written...


  Progress: 200/243 partitions written...


INFO:starepandas.ingest:✓ Stored 1C.GPM.GMI.XCAL2016-C.20250101-S204910-E222221.061578.V07B.HDF5 → ['s3://zarrpods/gmi-demo-parquet', 's3://zarrpods/gmi-demo-parquet']


INFO:starepandas.ingest:Ingested 2 Parquet dataset(s)


✓ Inserted 243 metadata rows into RDS
✓ Finished writing 243 Parquet partitions to s3://zarrpods/gmi-demo-parquet
GMI  : stored 2 dataset path(s)  (1C.GPM.GMI.XCAL2016-C.20250101-S204910-E222221.061578.V07B.HDF5)
CPU times: user 2min 35s, sys: 14.6 s, total: 2min 50s
Wall time: 15min 53s


## Step 2 — Find intersecting data via STARE SIDs

In [5]:
if BBOX is not None:
    location_sids = demo.get_sids_for_bbox(*BBOX, level=STARE_LEVEL)
    print(f"Generated {len(location_sids)} SIDs for bbox {BBOX}")
    intersecting = demo.find_intersecting_data(location_sids, instruments=["GMI"])
    # Scope to our granule. Substring match on the basename, which the flat
    # pod-code layout embeds in the chunk filename (bracketed by '-').
    if not intersecting.empty and "group_path" in intersecting.columns:
        intersecting = intersecting[
            intersecting["group_path"].str.contains(granule_path_marker, regex=False)
        ]
    print(f"Found {len(intersecting)} intersecting metadata row(s).")
else:
    location_sids = None
    intersecting = None
    print("BBOX is None — Step 4 will reconstitute the full granule directly.")

if intersecting is not None and not intersecting.empty:
    intersecting[["Dataset", "grouped_id", "group_path"]].head(8)


BBOX is None — Step 4 will reconstitute the full granule directly.


## Step 3 — Download intersecting Parquet partitions from S3

In [6]:
%%time
if intersecting is not None and not intersecting.empty:
    data_dict = demo.download_and_analyze(
        intersecting,
        instruments=list(intersecting["Dataset"].unique()),
    )
    for ds_name, sdf in data_dict.items():
        print(f"{ds_name}: {len(sdf)} rows, columns: {list(sdf.columns[:6])} …")
        display(sdf.head(3))
else:
    print("No intersecting partitions to download — Step 4 will read S3 directly.")
    data_dict = {}

No intersecting partitions to download — Step 4 will read S3 directly.
CPU times: user 85 μs, sys: 44 μs, total: 129 μs
Wall time: 124 μs


## Step 4 — Reconstitute HDF5 (S1 + S2)

In [7]:
%%time
# s3_prefix scope: with CLEAN_BEFORE_RUN=True the bucket only holds this
# granule's data, so passing the broad S3_PREFIX is correct and avoids the
# layout mismatch the old per-granule S3 prefix would create.
recon_path = demo.reconstitute_hdf5(
    dataset=DATASETS,
    output_hdf5_path=OUTPUT_HDF5,
    bbox=BBOX,
    s3_prefix=S3_PREFIX,
)
print(f"Written to: {recon_path}")


INFO:starepandas.demo_lib:Reconstituting HDF5 for dataset='GMI_S1' over full granule (no spatial filter)


INFO:starepandas.demo_lib:Reconstituting HDF5 for dataset='GMI_S2' over full granule (no spatial filter)


INFO:starepandas.demo_lib:✓ Reconstituted HDF5 written to /tmp/gmi_s3_reconstituted.h5


Written to: /tmp/gmi_s3_reconstituted.h5
CPU times: user 24.8 s, sys: 7.07 s, total: 31.9 s
Wall time: 4min 1s


## Step 5 — Structure comparison: reconstituted vs original

In [8]:
def dump_structure(path, label):
    """Print HDF5 group/dataset tree with shapes and dtypes."""
    print(f"\n--- {label} ---")
    with h5py.File(path, "r") as f:
        def _visit(name, obj):
            if isinstance(obj, h5py.Dataset):
                print(f"  /{name:50s} {str(obj.shape):20s} {obj.dtype}")
            elif isinstance(obj, h5py.Group) and name != "/":
                print(f"  /{name:50s} Group")
        f.visititems(_visit)

dump_structure(recon_path, f"RECONSTITUTED  ({os.path.basename(recon_path)})")
dump_structure(GRANULE_FILE, f"ORIGINAL       ({os.path.basename(GRANULE_FILE)})")


--- RECONSTITUTED  (gmi_s3_reconstituted.h5) ---
  /S1                                                 Group
  /S1/Latitude                                        (5966, 221)          float32
  /S1/Longitude                                       (5966, 221)          float32
  /S1/Quality                                         (5966, 221)          int8
  /S1/SCstatus                                        Group
  /S1/SCstatus/FractionalGranuleNumber                (5966,)              float64
  /S1/SCstatus/SCaltitude                             (5966,)              float32
  /S1/SCstatus/SClatitude                             (5966,)              float32
  /S1/SCstatus/SClongitude                            (5966,)              float32
  /S1/SCstatus/SCorientation                          (5966,)              int16
  /S1/ScanTime                                        Group
  /S1/ScanTime/DayOfMonth                             (5966,)              int8
  /S1/ScanTime/DayOfYear       

## Step 6 — RDS metadata verification

In [9]:
conn = _ensure_rds_db_and_table("StarePodsMetadata")
try:
    with conn.cursor() as cur:
        # Flat pod-code layout: the basename is embedded in the chunk
        # filename, so use a LIKE substring match (not a startswith prefix).
        cur.execute(
            'SELECT "Dataset", COUNT(*) '
            'FROM "PodsMetadata" '
            'WHERE "MetadataJson"->>%s LIKE %s '
            'GROUP BY "Dataset" ORDER BY "Dataset"',
            ("group_path", f"%{granule_path_marker}%"),
        )
        rows = cur.fetchall()
    print(f"RDS scope: group_path contains '{granule_path_marker}'")
    for ds, cnt in rows:
        print(f"  {ds}: {cnt} partition(s)")
finally:
    conn.close()


RDS scope: group_path contains '-1C.GPM.GMI.XCAL2016-C.20250101-S112952-E130304.061572.V07B-'
  GMI_S1: 265 partition(s)
  GMI_S2: 248 partition(s)


## Step 7 — Temporal catalog: every chunk carries `[t_start, t_end]` + podcode

`load_s3_temporal_catalog` returns the thin projection the analytics use (`podcode / Dataset / t_start / t_end`) from RDS — never the heavy `MetadataJson`. This reads the **shared** catalog filtered by instrument, so the counts include every GMI/SSMIS ingest in the table, not just this granule.

In [10]:
from starepandas.io.granules import (
    load_s3_metadata, load_s3_temporal_catalog, load_s3_vcf,
)
from starepandas.overlap import (
    rendezvous_events, overlap_matrix, overlap_pod_table, pair_drilldown,
    pod_drilldown,
)

catalog = pd.concat(
    [load_s3_temporal_catalog(dataset_prefix=instrument) for instrument in INSTRUMENTS],
    ignore_index=True,
)
print(f"Thin catalog ({'+'.join(INSTRUMENTS)}, catalog-wide): {len(catalog)} chunks across "
      f"{catalog['Dataset'].nunique()} datasets")
display(catalog.groupby('Dataset').agg(
    chunks=('podcode', 'size'),
    first_start=('t_start', 'min'),
    last_end=('t_end', 'max'),
))
catalog.head(6)

Thin catalog (GMI+SSMIS+AMSR2+ATMS, catalog-wide): 19912 chunks across 14 datasets


,chunks,first_start,last_end
Dataset,,,
AMSR2_S1,358,2025-01-01 20:19:15.458,2025-01-01 21:58:07.391
AMSR2_S2,357,2025-01-01 20:19:15.458,2025-01-01 21:58:07.391
AMSR2_S3,357,2025-01-01 20:19:15.458,2025-01-01 21:58:07.391
AMSR2_S4,358,2025-01-01 20:19:15.458,2025-01-01 21:58:07.391
AMSR2_S5,359,2025-01-01 20:19:15.458,2025-01-01 21:58:07.391
AMSR2_S6,356,2025-01-01 20:19:15.458,2025-01-01 21:58:07.391
ATMS_S1,502,2025-01-01 20:17:07.136,2025-01-01 21:58:35.136
ATMS_S2,501,2025-01-01 20:17:07.136,2025-01-01 21:58:35.136
GMI_S1,529,2025-01-01 11:29:53.002,2025-01-01 22:22:22.797


,podcode,Dataset,t_start,t_end
0,q30220,GMI_S1,2025-01-01 11:29:53.002,2025-01-01 11:30:53.002
1,q33110,GMI_S1,2025-01-01 11:29:53.002,2025-01-01 11:31:38.002
2,q33300,GMI_S1,2025-01-01 11:29:53.002,2025-01-01 11:31:17.377
3,q30223,GMI_S1,2025-01-01 11:30:11.752,2025-01-01 11:30:51.127
4,q33303,GMI_S1,2025-01-01 11:30:19.252,2025-01-01 11:31:15.502
5,q30221,GMI_S1,2025-01-01 11:30:34.252,2025-01-01 11:30:53.002


## Step 8 — Period-filtered load

`load_s3_metadata(..., period=(start, end))` keeps only chunks whose **data-level** range `[t_start, t_end]` overlaps the period (the shared `_period_conditions` index-friendly rewrite — live EXPLAIN confirms a Bitmap Index Scan on `idx_pods_temporal`). Two GMI passes are now ingested, ~9 h apart, so the filter can tell them apart: the same chunks, selected purely on their temporal range rather than on which granule they came from. A window days away returns none.

In [11]:
gmi = catalog[catalog['Dataset'].str.startswith('GMI')]
passes = {
    "first GMI pass ": gmi[gmi['t_start'] < PASS_SPLIT],
    "second GMI pass": gmi[gmi['t_start'] >= PASS_SPLIT],
}

for label, chunks in passes.items():
    window = (chunks['t_start'].min(), chunks['t_end'].max())
    hit = load_s3_metadata(dataset_prefix='GMI', period=window)
    print(f"{label}: [{window[0]}, {window[1]}]  ({len(chunks)} chunks)")
    print(f"  -> {len(hit)} of {len(gmi)} GMI chunks match this period")

miss_period = (PASS_SPLIT - pd.Timedelta(days=10), PASS_SPLIT - pd.Timedelta(days=9))
miss = load_s3_metadata(dataset_prefix='GMI', period=miss_period)
print(f"9-10 days earlier -> {len(miss)} chunks")

first GMI pass : [2025-01-01 11:29:53.002000, 2025-01-01 13:03:04.224000]  (513 chunks)
  -> 513 of 1020 GMI chunks match this period


second GMI pass: [2025-01-01 20:49:11.577000, 2025-01-01 22:22:22.797000]  (507 chunks)
  -> 507 of 1020 GMI chunks match this period


9-10 days earlier -> 0 chunks


## Step 9 — VCF temporal roll-up

`load_s3_vcf(level, ...)` groups chunks by their level-`level` ancestor pod and returns each pod's union range `[min(t_start), max(t_end)]` + child count — the on-the-fly temporal hierarchy ("Virtual Collection File"), nothing materialized.

In [12]:
vcf = load_s3_vcf(1, dataset_prefix='GMI')
print(f"{len(vcf)} level-1 VCF nodes for GMI (one per octant subtree)")
vcf

25 level-1 VCF nodes for GMI (one per octant subtree)


,podcode,t_start,t_end,n_chunks,n_without_range
0,q00,2025-01-01 22:00:47.179,2025-01-01 22:13:11.550,48,0
1,q01,2025-01-01 22:15:30.299,2025-01-01 22:19:56.548,20,0
2,q03,2025-01-01 22:04:47.178,2025-01-01 22:16:30.299,28,0
3,q12,2025-01-01 12:26:02.360,2025-01-01 12:39:09.856,55,0
4,q20,2025-01-01 12:37:07.982,2025-01-01 21:13:11.568,30,0
5,q21,2025-01-01 20:52:47.200,2025-01-01 20:58:17.199,20,0
6,q22,2025-01-01 12:55:24.851,2025-01-01 21:13:13.443,30,0
7,q23,2025-01-01 12:41:37.980,2025-01-01 21:13:11.568,104,0
8,q30,2025-01-01 11:29:53.002,2025-01-01 13:03:04.224,26,0
9,q31,2025-01-01 20:49:11.577,2025-01-01 22:22:22.797,64,0


## Step 10 — Multi-instrument overlap analytics

`rendezvous_events` sweeps for passes simultaneously present in a pod within Δt; the matrix / pod-table / drill-downs all aggregate that one events frame. Since the catalog-wide read (step 7) mixes every ingest in the shared RDS table, we scope the sweep to the **two windows this demo wrote** (the `period` pushes into SQL) so the result is this demo's genuine rendezvous.

With four instruments ingested, the slide-9 table gains its **n=3 and n=4** columns. The 4-way is real but tight: over pods `q03200`/`q03203` the passes arrive SSMIS 21:29 → AMSR2 21:46 → ATMS 21:47 → GMI 22:13, spanning ~45 min — so Δt has to be at least that wide before all four count as simultaneous. The first cell below shows that progression.

In [13]:
demo_catalog = pd.concat(
    [load_s3_temporal_catalog(dataset_prefix=instrument, period=window)
     for instrument in INSTRUMENTS for window in DEMO_WINDOWS],
    ignore_index=True,
)

print("How the coincidence window dt widens what counts as a rendezvous:")
for dt in (pd.Timedelta(minutes=15), pd.Timedelta(minutes=30), OVERLAP_DT):
    ev = rendezvous_events(demo_catalog, dt)
    table = overlap_pod_table(ev)
    by_n = {int(n): int(table[n].gt(0).sum()) for n in table.columns}
    print(f"  dt={str(dt).split()[-1]}  {len(ev):5d} events  "
          f"{ev['podcode'].nunique():4d} pods   pods by n-way: {by_n}")

events = rendezvous_events(demo_catalog, OVERLAP_DT)
pod_table = overlap_pod_table(events)
widest = max(pod_table.columns)

print(f"\nInstrument x instrument matrix — pods where A & B rendezvous (slide 8), dt={OVERLAP_DT}:")
display(overlap_matrix(events))

print('Per-pod n-way combination counts (slide 9), widest rendezvous first:')
display(pod_table.sort_values(sorted(pod_table.columns, reverse=True), ascending=False).head(10))

print(f"{int(pod_table[widest].gt(0).sum())} pods see all {widest} instruments; "
      f"{int(pod_table.get(3, pd.Series(dtype=int)).gt(0).sum())} see a 3-way.")

print('GMI-SSMIS pair drill-down (first 8 shared pods + crossing times):')
display(pair_drilldown(events, 'GMI', 'SSMIS').head(8))

pod = pod_table[pod_table[widest].gt(0)].index[0]
print(f'Pod drill-down for {pod} — every combination meeting there:')
display(pod_drilldown(events, pod).drop(columns='times'))

How the coincidence window dt widens what counts as a rendezvous:
  dt=00:15:00   1317 events   399 pods   pods by n-way: {2: 399, 3: 8}
  dt=00:30:00   1673 events   438 pods   pods by n-way: {2: 438, 3: 96}
  dt=00:45:00   1684 events   442 pods   pods by n-way: {2: 442, 3: 101, 4: 2}

Instrument x instrument matrix — pods where A & B rendezvous (slide 8), dt=0 days 00:45:00:


,AMSR2,ATMS,GMI,SSMIS
AMSR2,0,359,43,58
ATMS,359,0,68,81
GMI,43,68,0,47
SSMIS,58,81,47,0


Per-pod n-way combination counts (slide 9), widest rendezvous first:


n_instruments,2,3,4
podcode,,,
q03200,6,4,1
q03203,6,4,1
q00101,3,1,0
q00110,3,1,0
q00111,3,1,0
q00112,3,1,0
q00113,3,1,0
q00132,3,1,0
q00220,3,1,0


2 pods see all 4 instruments; 101 see a 3-way.
GMI-SSMIS pair drill-down (first 8 shared pods + crossing times):


,podcode,frequency,times
0,q01200,2,"[2025-01-01 22:15:32.174000, 2025-01-01 22:15:..."
1,q03200,2,"[2025-01-01 22:13:52.800000, 2025-01-01 22:14:..."
2,q03201,1,[2025-01-01 22:14:11.550000]
3,q03203,2,"[2025-01-01 22:13:39.675000, 2025-01-01 22:13:..."
4,q22200,4,"[2025-01-01 12:57:00.713000, 2025-01-01 12:57:..."
5,q22201,2,"[2025-01-01 12:56:51.101000, 2025-01-01 12:56:..."
6,q22202,2,"[2025-01-01 12:55:24.851000, 2025-01-01 12:55:..."
7,q22203,4,"[2025-01-01 12:56:20.840000, 2025-01-01 12:56:..."


Pod drill-down for q03200 — every combination meeting there:


,instruments,n_instruments,frequency
0,"(AMSR2, ATMS)",2,2
1,"(AMSR2, GMI)",2,2
2,"(AMSR2, SSMIS)",2,6
3,"(ATMS, GMI)",2,2
4,"(ATMS, SSMIS)",2,2
5,"(GMI, SSMIS)",2,2
6,"(AMSR2, ATMS, GMI)",3,2
7,"(AMSR2, ATMS, SSMIS)",3,2
8,"(AMSR2, GMI, SSMIS)",3,2
9,"(ATMS, GMI, SSMIS)",3,2
